# Lecture 2 — Learning Willmore-Minimising Surface Maps

**XXII EGD School, Piauí, Brazil 2026**

Based on the paper [*Minimising Willmore Energy via Neural Flow*](https://arxiv.org/abs/2604.04321)  
E. Hirst · H. N. Sá Earp · T. S. R. Silva

---

### What you will build

Starting from nothing but PyTorch and NumPy, you will train a small neural network to
learn the surface embedding that **minimises the Willmore energy** over all embedded tori.
The theoretical minimum is

$$W = 2\pi^2 \approx 19.74,$$

achieved by the **Clifford torus**.

### Roadmap

| Step | Topic |
|------|-------|
| 1 | Imports & setup |
| 2 | Sampling the parameter domain |
| 3 | Fourier features for periodicity |
| 4 | Initial torus embedding in $\mathbb{R}^3$ |
| 5 | 3D visualisation |
| 6 | Supervised pretraining |
| 7 | Willmore energy loss |
| 8 | Regularity loss |
| 9 | PINN training |
| 10 | Loss curves & surface evolution |

No prior ML knowledge is assumed — every concept is introduced as it appears.

## 1 — Imports and Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401 — registers 3-D projection

# Fix random seeds so results are reproducible
torch.manual_seed(42)
np.random.seed(42)

# Use CPU throughout (float32 is fine for this demo)
DEVICE = torch.device("cpu")
DTYPE  = torch.float32

PI = np.pi
TWO_PI = 2 * PI

print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")

## 2 — Sampling the Fundamental Domain

A **torus** $T^2$ is topologically a square with opposite edges identified:

$$T^2 \cong [0, 2\pi] \times [0, 2\pi] \big/ \sim$$

where $(0, v) \sim (2\pi, v)$ and $(u, 0) \sim (u, 2\pi)$.

We work in this **parameter space** and train a neural network to learn the map
$$\varphi : (u, v) \mapsto (x, y, z) \in \mathbb{R}^3.$$

For training we need a cloud of collocation points — random samples drawn uniformly
from $[0, 2\pi]^2$.  No mesh is required; this is the key advantage of PINNs over
classical finite-element methods.

In [ ]:
def sample_torus_domain(n: int) -> torch.Tensor:
    """
    Draw n points uniformly at random from [0, 2π] × [0, 2π].
    Returns a tensor of shape (n, 2).
    """
    return torch.rand(n, 2, dtype=DTYPE, device=DEVICE) * TWO_PI


# ── Visualise the domain ──────────────────────────────────────────────────────
N_SCATTER = 2000
uv = sample_torus_domain(N_SCATTER)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: random collocation cloud
ax = axes[0]
ax.scatter(uv[:, 0].numpy(), uv[:, 1].numpy(), s=2, alpha=0.4, color="steelblue")
ax.set_xlabel("$u$"); ax.set_ylabel("$v$")
ax.set_title(f"{N_SCATTER} random collocation points")
ax.set_aspect("equal")
# Draw identification boundaries
for x in [0, TWO_PI]:
    ax.axvline(x, color="crimson", lw=1.5, ls="--", label="identified" if x == 0 else "")
for y in [0, TWO_PI]:
    ax.axhline(y, color="navy", lw=1.5, ls="--")
ax.legend(fontsize=9)

# Right: uniform grid to show the square structure
ax = axes[1]
n_grid = 30
u_grid = np.linspace(0, TWO_PI, n_grid)
v_grid = np.linspace(0, TWO_PI, n_grid)
U, V = np.meshgrid(u_grid, v_grid)
ax.scatter(U.ravel(), V.ravel(), s=4, color="steelblue", alpha=0.6)
ax.set_xlabel("$u$"); ax.set_ylabel("$v$")
ax.set_title(f"Uniform {n_grid}×{n_grid} grid")
ax.set_aspect("equal")

plt.suptitle("Parameter domain  $[0, 2\\pi]^2$  —  opposite red/blue edges are identified", y=1.02)
plt.tight_layout()
plt.show()

print(f"Sample tensor shape: {uv.shape}   min: {uv.min():.3f}   max: {uv.max():.3f}")

## 3 — Fourier Feature Embeddings for Periodicity

### The problem

A standard multilayer perceptron (MLP) maps $\mathbb{R}^n \to \mathbb{R}^m$ and has no
built-in notion of periodicity.  If we feed raw $(u, v)$ as inputs, the network can
easily learn $\varphi(0, v) \neq \varphi(2\pi, v)$, violating the identification.

### The solution: Fourier features

Replace $(u, v)$ by its **Fourier feature vector**

$$\gamma(u, v) = \bigl(\sin u,\, \cos u,\, \sin v,\, \cos v,\;
                       \sin 2u,\, \cos 2u,\, \sin 2v,\, \cos 2v,\;
                       \ldots\bigr).$$

Because $\sin(k \cdot 0) = \sin(k \cdot 2\pi) = 0$ and similarly for cosine, the
feature vector is **identical** at identified boundary points.  Any downstream MLP
applied to $\gamma$ is therefore automatically periodic — no extra loss term required.

The number of frequencies $K$ controls expressiveness: $K = 1$ can only represent
simple shapes; larger $K$ is needed for fine surface detail.

In [ ]:
def fourier_features(uv: torch.Tensor, num_freqs: int = 4) -> torch.Tensor:
    """
    Map (u, v) → (sin u, cos u, sin v, cos v, sin 2u, cos 2u, …)
    for frequencies k = 1, …, num_freqs.

    Args:
        uv:        (N, 2) tensor with u = uv[:, 0],  v = uv[:, 1]
        num_freqs: number of frequency components K

    Returns:
        features:  (N, 4·K) tensor
    """
    parts = []
    for k in range(1, num_freqs + 1):
        parts.extend([
            torch.sin(k * uv[:, 0:1]),
            torch.cos(k * uv[:, 0:1]),
            torch.sin(k * uv[:, 1:2]),
            torch.cos(k * uv[:, 1:2]),
        ])
    return torch.cat(parts, dim=1)   # (N, 4·K)


# ── Verify periodicity ────────────────────────────────────────────────────────
N = 500
u_vals = torch.rand(N, 1, dtype=DTYPE) * TWO_PI    # random u ∈ [0, 2π]
v_vals = torch.rand(N, 1, dtype=DTYPE) * TWO_PI    # random v ∈ [0, 2π]

uv_left  = torch.cat([torch.zeros(N, 1, dtype=DTYPE), v_vals], dim=1)   # u = 0
uv_right = torch.cat([torch.full((N, 1), TWO_PI, dtype=DTYPE), v_vals], dim=1)  # u = 2π

feat_left  = fourier_features(uv_left,  num_freqs=4)
feat_right = fourier_features(uv_right, num_freqs=4)

max_diff = (feat_left - feat_right).abs().max().item()
print(f"Max |γ(0,v) − γ(2π,v)| over {N} points: {max_diff:.2e}  ✓" if max_diff < 1e-5
      else f"WARNING: periodicity not enforced, diff = {max_diff:.2e}")

# ── Feature dimension grows with K ────────────────────────────────────────────
for K in [1, 2, 4, 8]:
    f = fourier_features(uv[:4], num_freqs=K)
    print(f"  K={K}  →  feature dimension = {f.shape[1]}")

# ── Visualise the first two features for a 1-D slice (fixed v = π) ───────────
u_line = torch.linspace(0, TWO_PI, 300).unsqueeze(1)
v_line = torch.full_like(u_line, PI)
feat_line = fourier_features(torch.cat([u_line, v_line], dim=1), num_freqs=3)

fig, ax = plt.subplots(figsize=(9, 3))
labels = [r"$\sin u$", r"$\cos u$", r"$\sin 2u$", r"$\cos 2u$",
          r"$\sin 3u$", r"$\cos 3u$"]
for i in range(6):
    ax.plot(u_line.numpy(), feat_line[:, i].numpy(), label=labels[i], lw=1.5)
ax.axvline(0,      color="k", lw=0.8, ls=":")
ax.axvline(TWO_PI, color="k", lw=0.8, ls=":", label="identified boundaries")
ax.set_xlabel("$u$"); ax.set_title("Fourier features are periodic — $f(0) = f(2\\pi)$")
ax.legend(ncol=3, fontsize=9); plt.tight_layout(); plt.show()

## 4 — Initial Embedding into $\mathbb{R}^3$

The **standard torus** with major radius $R$ and tube radius $r$ is

$$\varphi(u, v) = \bigl((R + r\cos v)\cos u,\;(R + r\cos v)\sin u,\; r\sin v\bigr).$$

The **Willmore energy** of a smooth surface $\Sigma \hookrightarrow \mathbb{R}^3$ is

$$W(\varphi) = \iint_\Sigma H^2 \, dA,$$

where $H$ is the mean curvature and $dA = \sqrt{EG - F^2}\,du\,dv$ is the area element
computed from the first fundamental form coefficients
$E = \langle\varphi_u,\varphi_u\rangle$,
$F = \langle\varphi_u,\varphi_v\rangle$,
$G = \langle\varphi_v,\varphi_v\rangle$.

For a torus of revolution the energy has the closed form
$$W(R, r) = \frac{\pi}{r}\int_0^{2\pi}\frac{(R + 2r\cos v)^2}{R + r\cos v}\,dv,$$
which evaluates to $W = 4\pi^2/\!\sqrt{3} \approx 22.8$ for $(R,r)=(2,1)$ and to
the theoretical **minimum** $W = 2\pi^2 \approx 19.74$ for $R = \sqrt{2}$, $r = 1$
(the Clifford torus).

We use $(R, r) = (2, 1)$ as our **starting point** — the PINN will then drive $W$
down toward $2\pi^2$.

In [ ]:
# ── Analytic torus embedding ──────────────────────────────────────────────────
R_INIT, r_INIT = 2.0, 1.0   # starting torus (W ≈ 22.8)
R_CLIFF        = np.sqrt(2)  # Clifford torus (W = 2π²)

def torus_embed(uv: torch.Tensor, R: float = R_INIT, r: float = r_INIT) -> torch.Tensor:
    """
    Standard torus: φ(u,v) = ((R + r cos v) cos u, (R + r cos v) sin u, r sin v)

    Args:
        uv: (N, 2)  parameter coordinates u = uv[:,0], v = uv[:,1]
    Returns:
        xyz: (N, 3)
    """
    u, v = uv[:, 0], uv[:, 1]
    x = (R + r * v.cos()) * u.cos()
    y = (R + r * v.cos()) * u.sin()
    z = r * v.sin()
    return torch.stack([x, y, z], dim=1)


# Evaluate on a random sample
uv_sample = sample_torus_domain(4096)
xyz_init  = torus_embed(uv_sample)

print("Embedding shape :", xyz_init.shape)
print(f"x range : [{xyz_init[:,0].min():.2f}, {xyz_init[:,0].max():.2f}]")

# Verify periodicity: φ(0, v) == φ(2π, v)
v_test = torch.rand(200, dtype=DTYPE) * TWO_PI
uv_0   = torch.stack([torch.zeros_like(v_test), v_test], dim=1)
uv_2pi = torch.stack([torch.full_like(v_test, TWO_PI), v_test], dim=1)
max_gap = (torus_embed(uv_0) - torus_embed(uv_2pi)).abs().max().item()
print(f"Max |φ(0,v)−φ(2π,v)|: {max_gap:.2e}  ✓" if max_gap < 1e-5
      else f"Periodicity gap: {max_gap:.2e}")

## 5 — Visualising the Tori in 3D

We plot both the **starting torus** $(R=2, r=1)$ and the **target** Clifford torus
$(R=\sqrt{2}, r=1)$ side by side, colouring each surface by the analytic mean
curvature

$$H = -\frac{R + 2r\cos v}{2r(R + r\cos v)}.$$

This gives intuition about where curvature concentrates (the inner equator of the
tube) and how the two tori differ geometrically.

In [ ]:
def analytic_mean_curvature(U: np.ndarray, V: np.ndarray,
                            R: float, r: float) -> np.ndarray:
    """H = -(R + 2r cosv) / (2r(R + r cosv))  for a torus of revolution."""
    return -(R + 2 * r * np.cos(V)) / (2 * r * (R + r * np.cos(V)))


def plot_torus_3d(ax, R: float, r: float, title: str, n: int = 60) -> None:
    u = np.linspace(0, TWO_PI, n)
    v = np.linspace(0, TWO_PI, n)
    U, V = np.meshgrid(u, v)
    X = (R + r * np.cos(V)) * np.cos(U)
    Y = (R + r * np.cos(V)) * np.sin(U)
    Z = r * np.sin(V)
    H = analytic_mean_curvature(U, V, R, r)

    surf = ax.plot_surface(X, Y, Z, facecolors=plt.cm.coolwarm(
        (H - H.min()) / (H.max() - H.min())), alpha=0.90,
        linewidth=0, antialiased=True)
    ax.set_title(title, pad=10)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_box_aspect([1, 1, 0.5])
    # Add a proxy scalar bar
    sm = plt.cm.ScalarMappable(cmap="coolwarm",
                                norm=plt.Normalize(H.min(), H.max()))
    plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.12, label="$H$ (mean curvature)")


fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(121, projection="3d")
ax2 = fig.add_subplot(122, projection="3d")

plot_torus_3d(ax1, R=R_INIT,  r=r_INIT, title=f"Initial torus  $(R={R_INIT}, r={r_INIT})$")
plot_torus_3d(ax2, R=R_CLIFF, r=r_INIT, title=f"Clifford torus  $(R=\\sqrt{{2}},\\, r=1)$  — target")

plt.suptitle("Surface coloured by mean curvature $H$", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 6 — Supervised Pretraining

Before doing anything Willmore-related we **warm-start** the network to reproduce
the analytic embedding $\varphi_{\text{init}}(u,v)$ using plain supervised learning.

**Why?** If we started PINN training from a random network, the output would be a
chaotic cloud that is nowhere near a smooth surface.  Pretraining gives us a sensible
surface (the starting torus) as the initial condition for the physics training.

**Architecture:** `Fourier features → Linear(64) → Tanh → Linear(128) → Tanh → Linear(64) → Tanh → Linear(3)`.

**Loss:** mean squared error $\mathcal{L}_\text{sup} = \frac{1}{N}\sum_i \|\hat\varphi_i - \varphi_i\|^2$.

In [ ]:
# ── Network definition ────────────────────────────────────────────────────────

NUM_FREQS = 4    # Fourier frequencies per dimension

class TorusNet(nn.Module):
    """
    Periodic MLP:  (u,v) → Fourier features → tanh MLP → (x, y, z)

    The Fourier feature layer enforces φ(0,v) = φ(2π,v) and φ(u,0) = φ(u,2π)
    by construction, so no periodicity penalty is needed in the loss.
    """
    def __init__(self, num_freqs: int = NUM_FREQS,
                 hidden: list = [64, 128, 128, 64]):
        super().__init__()
        in_dim = 4 * num_freqs
        self.num_freqs = num_freqs

        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            prev = h
        layers += [nn.Linear(prev, 3)]          # output: (x, y, z)
        self.net = nn.Sequential(*layers)

        # Xavier initialisation — good default for tanh networks
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, uv: torch.Tensor) -> torch.Tensor:
        return self.net(fourier_features(uv, self.num_freqs))


# ── Supervised training loop ──────────────────────────────────────────────────

def supervised_train(model: nn.Module,
                     R: float = R_INIT, r: float = r_INIT,
                     epochs: int = 300,
                     lr: float = 1e-3,
                     batch: int = 512) -> list:
    """Train model to match the analytic torus via MSE on random batches."""
    opt = optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(1, epochs + 1):
        uv  = sample_torus_domain(batch)
        xyz = torus_embed(uv, R, r)            # ground-truth targets

        pred = model(uv)
        loss = nn.functional.mse_loss(pred, xyz)

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(loss.item())
        if epoch % 60 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{epochs}  supervised MSE = {loss.item():.5f}")
    return losses


torch.manual_seed(0)
model = TorusNet().to(DEVICE)

print(f"Network parameters: {sum(p.numel() for p in model.parameters()):,}")
print("── Supervised pretraining ──")
sup_losses = supervised_train(model, epochs=300)

# ── Verify the network matches the torus ─────────────────────────────────────
with torch.no_grad():
    uv_check = sample_torus_domain(2000)
    err = (model(uv_check) - torus_embed(uv_check)).norm(dim=1).mean().item()
print(f"\nMean reconstruction error (L2 per point): {err:.4f}")

## 7 — The Willmore Energy Loss

### Differential geometry via autograd

We need $H$ and $dA$ at each collocation point.  Rather than deriving closed-form
expressions, we let **PyTorch's automatic differentiation** compute them from the
network output $\varphi$.

The recipe is:

1. **First fundamental form** — evaluate $\varphi_u = \partial\varphi/\partial u$
   and $\varphi_v = \partial\varphi/\partial v$ with `torch.autograd.grad`.

   $$E = \langle\varphi_u,\varphi_u\rangle,\quad
     F = \langle\varphi_u,\varphi_v\rangle,\quad
     G = \langle\varphi_v,\varphi_v\rangle, \quad
     dA = \sqrt{EG - F^2}.$$

2. **Unit normal** — $\hat{n} = (\varphi_u \times \varphi_v)/|\varphi_u \times \varphi_v|$.

3. **Second fundamental form** — differentiate $\varphi_u$ and $\varphi_v$ once more.

   $$L = \langle\varphi_{uu},\hat{n}\rangle,\quad
     M = \langle\varphi_{uv},\hat{n}\rangle,\quad
     N = \langle\varphi_{vv},\hat{n}\rangle.$$

4. **Mean curvature** —
   $H = \dfrac{EN - 2FM + GL}{2(EG - F^2)}.$

5. **Monte Carlo Willmore energy** —
   $W \approx (2\pi)^2\,\overline{H^2 \cdot dA},$
   where the bar denotes a batch average.

In [ ]:
EPS = 1e-8   # numerical floor for denominators / norms


def _first_derivs(model: nn.Module, uv: torch.Tensor):
    """
    Compute φ(uv), φ_u, φ_v via autograd.

    We loop over the 3 output components and compute the gradient of each
    w.r.t. the 2-dimensional input.  This gives a (N,3) tensor for each
    partial derivative while keeping the computation graph intact for the
    second derivatives we need below.

    Returns: phi (N,3), phi_u (N,3), phi_v (N,3), uv_leaf (N,2)
    """
    uv_leaf = uv.detach().requires_grad_(True)
    phi = model(uv_leaf)                        # (N, 3)

    phi_u_cols, phi_v_cols = [], []
    for k in range(3):
        g = torch.autograd.grad(phi[:, k].sum(), uv_leaf,
                                create_graph=True, retain_graph=True)[0]  # (N,2)
        phi_u_cols.append(g[:, 0:1])
        phi_v_cols.append(g[:, 1:2])

    phi_u = torch.cat(phi_u_cols, dim=1)   # (N, 3)
    phi_v = torch.cat(phi_v_cols, dim=1)   # (N, 3)
    return phi, phi_u, phi_v, uv_leaf


def _second_derivs(phi_u: torch.Tensor, phi_v: torch.Tensor, uv_leaf: torch.Tensor):
    """
    Compute φ_uu, φ_uv, φ_vv by differentiating φ_u and φ_v once more.
    Returns: phi_uu, phi_uv, phi_vv  each of shape (N, 3)
    """
    uu, uv_cols, vv = [], [], []
    for k in range(3):
        g_u = torch.autograd.grad(phi_u[:, k].sum(), uv_leaf,
                                  create_graph=True, retain_graph=True)[0]
        uu.append(g_u[:, 0:1])
        uv_cols.append(g_u[:, 1:2])

        g_v = torch.autograd.grad(phi_v[:, k].sum(), uv_leaf,
                                  create_graph=True, retain_graph=True)[0]
        vv.append(g_v[:, 1:2])

    return (torch.cat(uu, dim=1),
            torch.cat(uv_cols, dim=1),
            torch.cat(vv, dim=1))


def compute_willmore(model: nn.Module, uv_batch: torch.Tensor) -> torch.Tensor:
    """
    Monte Carlo estimate  W ≈ (2π)² · mean(H² · √(EG−F²))
    over a batch of collocation points drawn from [0, 2π]².
    Returns a differentiable scalar tensor.
    """
    _, phi_u, phi_v, uv_leaf = _first_derivs(model, uv_batch)

    # First fundamental form
    E     = (phi_u * phi_u).sum(1)
    F_coef = (phi_u * phi_v).sum(1)
    G     = (phi_v * phi_v).sum(1)
    det   = torch.clamp(E * G - F_coef ** 2, min=EPS)

    # Unit normal
    n_hat = torch.linalg.cross(phi_u, phi_v)
    n_hat = n_hat / n_hat.norm(dim=1, keepdim=True).clamp(min=EPS)

    # Second fundamental form
    phi_uu, phi_uv, phi_vv = _second_derivs(phi_u, phi_v, uv_leaf)
    L  = (phi_uu * n_hat).sum(1)
    M  = (phi_uv * n_hat).sum(1)
    N_ = (phi_vv * n_hat).sum(1)

    # Mean curvature and area element
    H      = (E * N_ - 2 * F_coef * M + G * L) / (2 * det)
    area_el = det.sqrt()

    return torch.mean(H ** 2 * area_el) * TWO_PI ** 2


# ── Evaluate on the pretrained (= initial) surface ───────────────────────────
uv_eval = sample_torus_domain(3000)
W_init  = compute_willmore(model, uv_eval)

print(f"Willmore energy on initial surface  W = {W_init.item():.4f}")
print(f"Analytic value for (R=2, r=1)       W = 4π²/√3 ≈ {4*PI**2/np.sqrt(3):.4f}")
print(f"Target (Clifford torus)             W = 2π²    ≈ {2*PI**2:.4f}")

## 8 — Regularity Loss

Minimising the Willmore energy alone can cause the parametrisation to **degenerate**:
the network may try to collapse parts of the torus (area element $\to 0$) while
keeping $W$ finite.  This is the parametric analogue of a mesh folding over itself.

We prevent this with a **regularity loss** that penalises vanishing area elements:

$$\mathcal{L}_R = \frac{1}{N}\sum_i \max\!\bigl(0,\;\delta - \sqrt{E_i G_i - F_i^2}\bigr)^2,$$

where $\delta > 0$ is a small threshold (we use $\delta = 0.05$).  This is zero
whenever the parametrisation is non-degenerate and fires only when some region of
the surface is being compressed.

A well-conditioned torus of revolution has area element $r(R + r\cos v)$, which for
$(R=2, r=1)$ ranges from $r(R-r) = 1$ to $r(R+r) = 3$ — comfortably above $\delta$.

In [ ]:
MIN_AREA = 0.05    # collapse threshold δ


def compute_regularity(model: nn.Module, uv_batch: torch.Tensor,
                       min_area: float = MIN_AREA) -> torch.Tensor:
    """
    Penalise vanishing area element  √(EG − F²) < min_area.

    L_R = mean( relu(min_area − √(EG−F²))² )
    """
    _, phi_u, phi_v, _ = _first_derivs(model, uv_batch)

    E      = (phi_u * phi_u).sum(1)
    F_coef = (phi_u * phi_v).sum(1)
    G      = (phi_v * phi_v).sum(1)
    det    = torch.clamp(E * G - F_coef ** 2, min=EPS)
    area_el = det.sqrt()

    return torch.nn.functional.relu(min_area - area_el).pow(2).mean()


# ── Evaluate on the initial surface ──────────────────────────────────────────
L_R = compute_regularity(model, uv_eval)
print(f"Regularity loss on initial surface  L_R = {L_R.item():.6f}")
print("(Should be ≈ 0 because the initial surface is non-degenerate.)")

# Illustrate: what happens if we force collapse?
with torch.no_grad():
    v_slice = torch.zeros(500, 2, dtype=DTYPE)   # only vary u, fix v=0
    v_slice[:, 0] = torch.linspace(0, TWO_PI, 500)

    phi_u_vals = torch.stack([
        torch.autograd.grad(model(v_slice)[:, k].sum(), v_slice,
                            retain_graph=True, create_graph=False)[0][:, 0]
        for k in range(3)
    ], dim=1)   # (500, 3)  — only approximate for illustration

# Show the distribution of area elements on the initial surface
_, phi_u_all, phi_v_all, _ = _first_derivs(model, uv_eval)
E_all   = (phi_u_all * phi_u_all).sum(1).detach()
F_all   = (phi_u_all * phi_v_all).sum(1).detach()
G_all   = (phi_v_all * phi_v_all).sum(1).detach()
area_el_all = (E_all * G_all - F_all**2).clamp(min=EPS).sqrt()

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(area_el_all.numpy(), bins=50, color="steelblue", edgecolor="white")
ax.axvline(MIN_AREA, color="crimson", ls="--", label=f"threshold δ = {MIN_AREA}")
ax.set_xlabel("$\\sqrt{EG - F^2}$  (area element)")
ax.set_ylabel("count")
ax.set_title("Distribution of area elements — initial surface")
ax.legend(); plt.tight_layout(); plt.show()

## 9 — PINN Training

We now minimise the **combined loss**

$$\mathcal{L} = \lambda_W\,\mathcal{L}_W + \lambda_R\,\mathcal{L}_R$$

with $\lambda_W = 1.0$ and $\lambda_R = 5.0$.

Key choices:

| Hyperparameter | Value | Reason |
|---------------|-------|--------|
| Optimiser | Adam | Works well for loss landscapes with scale separation |
| Learning rate | $3 \times 10^{-5}$ | Small enough to follow the energy surface smoothly |
| Batch size | 2 000 | Balance between gradient noise and per-step cost |
| Epochs | 500 | Enough to see a clear descent; ~2–3 min on CPU |
| Resample each epoch | yes | Fresh collocation points avoid overfitting to a fixed grid |

Surface snapshots are saved every 100 epochs so we can watch the shape evolve.

In [ ]:
# ── Training hyperparameters ──────────────────────────────────────────────────
PINN_EPOCHS    = 500
PINN_LR        = 3e-5
PINN_BATCH     = 2000
LAMBDA_W       = 1.0    # Willmore weight
LAMBDA_R       = 5.0    # regularity weight
SNAPSHOT_EVERY = 100    # save surface snapshot every N epochs

# ── Grid used for surface snapshots ──────────────────────────────────────────
N_GRID = 50
u_grid = np.linspace(0, TWO_PI, N_GRID)
v_grid = np.linspace(0, TWO_PI, N_GRID)
Ug, Vg = np.meshgrid(u_grid, v_grid)
uv_grid = torch.tensor(
    np.stack([Ug.ravel(), Vg.ravel()], axis=1), dtype=DTYPE, device=DEVICE
)

def surface_snapshot(model: nn.Module) -> np.ndarray:
    """Return (N_GRID², 3) numpy array of the current surface on the grid."""
    with torch.no_grad():
        return model(uv_grid).numpy()

# ── Training ──────────────────────────────────────────────────────────────────
optimizer = optim.Adam(model.parameters(), lr=PINN_LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PINN_EPOCHS, eta_min=1e-7)

history_W, history_R = [], []
snapshots = {}   # epoch → (N_GRID², 3)

# Save initial snapshot
snapshots[0] = surface_snapshot(model)

print(f"{'Epoch':>6}  {'W (Willmore)':>14}  {'L_R (regularity)':>18}  {'LR':>10}")
print("-" * 56)

for epoch in range(1, PINN_EPOCHS + 1):
    uv_batch = sample_torus_domain(PINN_BATCH)

    # ── Combined forward / backward ───────────────────────────────────────────
    # We share the first-derivative computation between both losses to avoid
    # recomputing the Jacobian twice.
    uv_leaf = uv_batch.detach().requires_grad_(True)
    phi = model(uv_leaf)

    # First derivatives (shared)
    phi_u_cols, phi_v_cols = [], []
    for k in range(3):
        g = torch.autograd.grad(phi[:, k].sum(), uv_leaf,
                                create_graph=True, retain_graph=True)[0]
        phi_u_cols.append(g[:, 0:1])
        phi_v_cols.append(g[:, 1:2])
    phi_u = torch.cat(phi_u_cols, dim=1)
    phi_v = torch.cat(phi_v_cols, dim=1)

    E_b     = (phi_u * phi_u).sum(1)
    F_b     = (phi_u * phi_v).sum(1)
    G_b     = (phi_v * phi_v).sum(1)
    det_b   = torch.clamp(E_b * G_b - F_b ** 2, min=EPS)
    area_el = det_b.sqrt()

    # Regularity loss (uses only first derivatives)
    L_R = torch.nn.functional.relu(MIN_AREA - area_el).pow(2).mean()

    # Second derivatives for Willmore
    n_hat = torch.linalg.cross(phi_u, phi_v)
    n_hat = n_hat / n_hat.norm(dim=1, keepdim=True).clamp(min=EPS)

    phi_uu_cols, phi_uv_cols, phi_vv_cols = [], [], []
    for k in range(3):
        g2u = torch.autograd.grad(phi_u[:, k].sum(), uv_leaf,
                                  create_graph=True, retain_graph=True)[0]
        phi_uu_cols.append(g2u[:, 0:1])
        phi_uv_cols.append(g2u[:, 1:2])
        g2v = torch.autograd.grad(phi_v[:, k].sum(), uv_leaf,
                                  create_graph=True, retain_graph=True)[0]
        phi_vv_cols.append(g2v[:, 1:2])

    phi_uu = torch.cat(phi_uu_cols, dim=1)
    phi_uv = torch.cat(phi_uv_cols, dim=1)
    phi_vv = torch.cat(phi_vv_cols, dim=1)

    L_ff = (phi_uu * n_hat).sum(1)
    M_ff = (phi_uv * n_hat).sum(1)
    N_ff = (phi_vv * n_hat).sum(1)
    H    = (E_b * N_ff - 2 * F_b * M_ff + G_b * L_ff) / (2 * det_b)

    L_W = torch.mean(H ** 2 * area_el) * TWO_PI ** 2

    loss = LAMBDA_W * L_W + LAMBDA_R * L_R

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    history_W.append(L_W.item())
    history_R.append(L_R.item())

    if epoch % 50 == 0:
        lr_now = scheduler.get_last_lr()[0]
        print(f"{epoch:6d}  {L_W.item():14.4f}  {L_R.item():18.6f}  {lr_now:.2e}")

    if epoch % SNAPSHOT_EVERY == 0:
        snapshots[epoch] = surface_snapshot(model)

print("\nTraining complete.")
print(f"Final Willmore estimate : {history_W[-1]:.4f}")
print(f"Target  (2π²)           : {2 * PI**2:.4f}")

## 10 — Loss Curves and Surface Evolution

We now visualise:

1. **Loss curves** — Willmore energy and regularity loss vs. epoch.
2. **Surface snapshots** — the network's output surface at epochs 0, 100, 200, 300, 400, 500,
   showing how the shape deforms toward the Willmore minimiser.
3. **Final energy** — compare the PINN result to the theoretical value $W = 2\pi^2$.

In [ ]:
# ── Loss curves ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs_ax = np.arange(1, PINN_EPOCHS + 1)

ax = axes[0]
ax.plot(epochs_ax, history_W, color="steelblue", lw=1.5, label="Willmore $W$")
ax.axhline(2 * PI**2, color="crimson", ls="--", lw=1.5, label=f"Target $2\\pi^2 = {2*PI**2:.2f}$")
ax.set_xlabel("Epoch"); ax.set_ylabel("$W$")
ax.set_title("Willmore energy vs. epoch")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.semilogy(epochs_ax, history_R, color="darkorange", lw=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("$\\mathcal{L}_R$  (log scale)")
ax.set_title("Regularity loss vs. epoch")
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

# ── Surface evolution snapshots ───────────────────────────────────────────────
snap_epochs = sorted(snapshots.keys())
n_snaps = len(snap_epochs)

fig = plt.figure(figsize=(4 * n_snaps, 4))
for i, ep in enumerate(snap_epochs):
    xyz = snapshots[ep].reshape(N_GRID, N_GRID, 3)
    ax  = fig.add_subplot(1, n_snaps, i + 1, projection="3d")
    ax.plot_surface(xyz[:, :, 0], xyz[:, :, 1], xyz[:, :, 2],
                    color="steelblue", alpha=0.80, linewidth=0, antialiased=True)
    W_ep = history_W[ep - 1] if ep > 0 else history_W[0]
    ax.set_title(f"Epoch {ep}\n$W={W_ep:.2f}$" if ep > 0 else
                 f"Epoch 0\n(pretrained)", fontsize=9)
    ax.set_xlabel("x", fontsize=7); ax.set_ylabel("y", fontsize=7)
    ax.set_zlabel("z", fontsize=7)
    ax.set_box_aspect([1, 1, 0.5])
    ax.tick_params(labelsize=6)

plt.suptitle("Surface evolution during PINN training", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

# ── Final comparison ──────────────────────────────────────────────────────────
W_final_mc = compute_willmore(model, sample_torus_domain(8000)).item()
W_target   = 2 * PI**2

print("─" * 50)
print(f"Initial Willmore  (R=2, r=1)  :  4π²/√3  ≈ {4*PI**2/np.sqrt(3):.4f}")
print(f"Final PINN estimate           :           {W_final_mc:.4f}")
print(f"Theoretical minimum (Clifford):  2π²     ≈ {W_target:.4f}")
print(f"Gap to minimum                :  {W_final_mc - W_target:.4f}  ({100*(W_final_mc/W_target-1):.1f} %)")